# Breakout DQN smoke test — Nature 2015 (Google Colab)

[Open in Google Colab](https://colab.research.google.com/github/benphuang/atari/blob/main/atari_nature2015.ipynb).

Select a GPU runtime if available. Run the cells in order. Setup installs dependencies into Colab's active Python and clones the project; the training cell uses the `nature2015` preset with a bounded 2,000-step smoke configuration. Core algorithm settings come from the preset; replay capacity, learning start, and evaluation duration are reduced for this smoke run. This notebook does not use bash or shell cells.

**ROM terms:** ROM installation is a separate, interactive step. Review and accept the AutoROM license before entering `ACCEPT`. Colab's `/content` storage is temporary.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/benphuang/atari.git"
PROJECT_DIR = Path("/content/atari")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it and retry.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)

# Keep Colab's preinstalled NumPy and PyTorch builds.
packages = [
    "ale-py>=0.11",
    "AutoROM>=0.6",
    "gymnasium>=1.0,<2",
    "opencv-python-headless>=4.8",
    "tensorboard>=2.14",
    "tyro>=0.8",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import torch
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU unavailable; the smoke run will use CPU.")

## Install Atari ROMs

Review the AutoROM license before running this cell. The installation did not auto-accept the license.

In [ ]:
import shutil

if input("After reviewing the AutoROM license, type ACCEPT to download ROMs: ").strip() != "ACCEPT":
    raise RuntimeError("ROM installation skipped; accept the license before rerunning this cell.")
autorom = shutil.which("AutoROM")
if autorom is None:
    raise RuntimeError("AutoROM executable was not installed in this runtime.")
subprocess.run([autorom, "--accept-license"], check=True)

In [ ]:
import os
import sys

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import ale_py
import gymnasium as gym

gym.register_envs(ale_py)
env = gym.make("BreakoutNoFrameskip-v4")
observation, _ = env.reset(seed=1)
assert observation.ndim == 3 and observation.shape[-1] == 3, f"Unexpected observation shape: {observation.shape}"
print(f"Breakout preflight passed: {observation.shape}, {env.action_space.n} actions")
env.close()

## Run the Nature 2015 smoke test

The preset's network, loss, optimizer, target network, preprocessing, frame-skip, and exploration settings are preserved. Smoke-only overrides cap training at 2,000 steps, shrink replay, start updates earlier, use frame-based replay for lower memory use, and shorten final evaluation.

In [ ]:
import os
import runpy
import sys
from pathlib import Path

import torch

PROJECT_DIR = Path("/content/atari").resolve()
TRAINING_SCRIPT = PROJECT_DIR / "atari_mnih.py"
if not TRAINING_SCRIPT.is_file():
    raise FileNotFoundError(f"{TRAINING_SCRIPT} is missing. Run the setup cell first.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)

EXP_NAME = "colab_nature2015_breakout_smoke"
TOTAL_TIMESTEPS = 2000
smoke_args = [
    "--preset", "nature2015",
    "--exp-name", EXP_NAME,
    "--seed", "1",
    "--env-id", "BreakoutNoFrameskip-v4",
    "--total-timesteps", str(TOTAL_TIMESTEPS),
    "--buffer-size", "5000",
    "--batch-size", "32",
    "--learning-starts", "500",
    "--exploration-steps", str(TOTAL_TIMESTEPS),
    "--eval-frequency", "0",
    "--eval-steps", "250",
    "--heldout-frequency", "0",
    "--replay-storage", "frames",
]
if not torch.cuda.is_available():
    smoke_args.append("--no-cuda")

print(f"Preset: nature2015 | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
previous_argv = sys.argv
try:
    sys.argv = [str(TRAINING_SCRIPT), *smoke_args]
    runpy.run_path(str(TRAINING_SCRIPT), run_name="__main__")
finally:
    sys.argv = previous_argv

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

event_files = sorted(Path("runs").glob(f"BreakoutNoFrameskip-v4__{EXP_NAME}__1__*/events.out.tfevents.*"))
assert event_files, "No TensorBoard event file was written."
events = EventAccumulator(str(event_files[-1])).Reload()
assert "eval/mean_return" in events.Tags()["scalars"], f"Missing final evaluation metric: {events.Tags()}"
evaluation = events.Scalars("eval/mean_return")[-1]
assert evaluation.step == TOTAL_TIMESTEPS, f"Expected final evaluation at {TOTAL_TIMESTEPS}; got {evaluation.step}"
print(f"Nature 2015 Breakout smoke test passed: mean return={evaluation.value:.2f} at step {evaluation.step}")